# 04e. Veto de mãe

Censo sem CPF no [`04_atribuir.ipynb`](04_atribuir.ipynb) cujo **melhor par**
cai no veto de mãe: as duas mães preenchidas, nem iguais nem prefixo
(`n ≥ 2` no começo), e não é o caso de nome completo e data iguais.

O [`04c_decompor_sem_cpf.ipynb`](04c_decompor_sem_cpf.ipynb) mistura isso com
primeiro nome ausente ou com mais de 1 de Levenshtein. Aqui os dois ficam
separados. A escada já aceita Jaro-Winkler da mãe ≥ 0,75, e de 0,95 a 0,99
não aplica esses vetos. Este notebook continua mostrando o corte estrito.


In [1]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    SPLINK_ATRIBUICAO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

print_paths()
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b antes)')
require_input(SPLINK_ATRIBUICAO, label='SPLINK_ATRIBUICAO (rode o 04_atribuir antes)')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

('censo_limpo_aplicacao', 'cpf_limpo_aplicacao')

In [2]:
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao_2 AS
SELECT unique_id_censo
FROM read_parquet('{SPLINK_ATRIBUICAO}')
''')

print(
    'Pares no parquet:',
    f"{con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]:,}",
)
print(
    'Atribuições:',
    f"{con.execute('SELECT COUNT(*) FROM atribuicao_2').fetchone()[0]:,}",
)


Pares no parquet: 6,598,211
Atribuições: 3,893,989


Melhor par de quem não saiu no 04. `veto_mae` exige primeiro
nome preenchido e com no máximo 1 de Levenshtein: se o primeiro nome já
reprova, o par vai para `so_primeiro` e não entra na leitura da mãe.


In [3]:
tok_c = "string_split(ca.nome_mae_phon, ' ')"
tok_p = "string_split(pb.nome_mae_phon, ' ')"
n_mae = f"least(len({tok_c}), len({tok_p}))"
prefixo_mae = (
    f"{n_mae} >= 2 AND "
    f"list_slice({tok_c}, 1, {n_mae}) = list_slice({tok_p}, 1, {n_mae})"
)
# Nome completo e data iguais passam mesmo com mãe diferente.
excecao = (
    "ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL "
    "AND ca.nome_completo_phon = pb.nome_completo_phon "
    "AND ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento = pb.data_nascimento"
)

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
      SELECT 1 FROM atribuicao_2 a WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute('''
CREATE OR REPLACE TABLE melhor_par AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM splink_predictions
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY unique_id_censo
    ORDER BY match_probability DESC, unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_sem AS
SELECT
    s.unique_id AS unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    CASE
        WHEN p.unique_id_cpf IS NULL THEN 'sem_par'
        WHEN ca.primeiro_nome_phon IS NULL
             OR pb.primeiro_nome_phon IS NULL
             OR levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) > 1
            THEN 'so_primeiro'
        WHEN ca.nome_mae_phon IS NOT NULL
             AND pb.nome_mae_phon IS NOT NULL
             AND ca.nome_mae_phon <> pb.nome_mae_phon
             AND NOT ({prefixo_mae})
             AND NOT ({excecao})
            THEN 'veto_mae'
        ELSE 'passaria'
    END AS porta
FROM censo_sem_cpf s
LEFT JOIN melhor_par p ON p.unique_id_censo = s.unique_id
LEFT JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = s.unique_id
LEFT JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
''')

display(con.execute('''
SELECT
    porta,
    COUNT(*) AS n,
    SUM(CASE WHEN match_probability >= 0.75 THEN 1 ELSE 0 END) AS n_p075,
    ROUND(AVG(match_probability), 3) AS avg_p
FROM melhor_sem
GROUP BY 1
ORDER BY n DESC
''').df())


,porta,n,n_p075,avg_p
0,sem_par,1463222,0.0,NaN
1,passaria,565268,260914.0,0.562
2,veto_mae,79652,19802.0,0.381
3,so_primeiro,76899,64576.0,0.840


No veto de mãe, como os dois nomes da mãe se afastam, ao lado
do nome da pessoa e da data. `mesmas_palavras_ordem` e `contem_sem_prefixo`
são o lugar em que a comparação pode estar rígida demais: as palavras batem,
a regra de prefixo não.


In [4]:
tok_c = "string_split(ca.nome_mae_phon, ' ')"
tok_p = "string_split(pb.nome_mae_phon, ' ')"
n_c = f"len({tok_c})"
n_p = f"len({tok_p})"
inter = f"len(list_intersect({tok_c}, {tok_p}))"

con.execute(f'''
CREATE OR REPLACE TABLE veto_mae AS
SELECT
    m.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    ca.nome_completo_phon AS nome_censo,
    pb.nome_completo_phon AS nome_cpf,
    ca.nome_mae_phon AS mae_censo,
    pb.nome_mae_phon AS mae_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) AS jw_mae,
    CASE
        WHEN list_sort({tok_c}) = list_sort({tok_p}) AND {tok_c} <> {tok_p}
            THEN 'mesmas_palavras_ordem'
        WHEN {tok_c} = {tok_p}
            THEN 'mesmas_palavras'
        WHEN {inter} = least({n_c}, {n_p}) AND {n_c} <> {n_p}
            THEN 'contem_sem_prefixo'
        WHEN jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) >= 0.92
            THEN 'parecidas'
        WHEN {inter} >= 1
            THEN 'alguma_palavra'
        ELSE 'sem_palavra'
    END AS mae_dif,
    CASE
        WHEN ca.nome_completo_phon = pb.nome_completo_phon THEN 'exato'
        WHEN ca.primeiro_nome_phon = pb.primeiro_nome_phon
             AND ca.ultimo_nome_phon = pb.ultimo_nome_phon THEN 'pontas'
        ELSE 'outro'
    END AS nome_pessoa,
    CASE
        WHEN ca.data_nascimento IS NULL AND pb.data_nascimento IS NULL THEN 'as_duas_nulas'
        WHEN ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL THEN 'uma_nula'
        WHEN ca.data_nascimento = pb.data_nascimento THEN 'exata'
        ELSE 'diferente'
    END AS data_par
FROM melhor_sem m
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE m.porta = 'veto_mae'
''')

n_excecao = con.execute('''
SELECT COUNT(*) FROM veto_mae
WHERE nome_pessoa = 'exato' AND data_par = 'exata'
''').fetchone()[0]
print('Veto de mãe:', f"{con.execute('SELECT COUNT(*) FROM veto_mae').fetchone()[0]:,}")
print('Nome e data exatos dentro do veto (tem de ser 0):', n_excecao)

display(con.execute('''
SELECT
    mae_dif,
    nome_pessoa,
    data_par,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p,
    ROUND(AVG(jw_mae), 3) AS avg_jw_mae
FROM veto_mae
GROUP BY 1, 2, 3
ORDER BY n DESC
''').df())


Veto de mãe: 79,652
Nome e data exatos dentro do veto (tem de ser 0): 0


,mae_dif,nome_pessoa,data_par,n,pct,avg_p,avg_jw_mae
0,alguma_palavra,exato,diferente,27472,34.5,0.315,0.679
1,alguma_palavra,exato,uma_nula,14325,18.0,0.323,0.676
2,sem_palavra,exato,diferente,9186,11.5,0.389,0.594
3,sem_palavra,exato,uma_nula,6133,7.7,0.396,0.589
4,alguma_palavra,outro,exata,3522,4.4,0.829,0.674
5,sem_palavra,outro,exata,3062,3.8,0.825,0.595
6,alguma_palavra,pontas,exata,1998,2.5,0.768,0.669
7,parecidas,exato,diferente,1943,2.4,0.100,0.961
8,sem_palavra,pontas,exata,1892,2.4,0.757,0.601
9,alguma_palavra,pontas,diferente,1682,2.1,0.161,0.692


Amostra. Primeiro as mães com as mesmas palavras ou uma dentro
da outra. Depois as parecidas com nome da pessoa exato. Por último, mãe sem
palavra em comum e nome da pessoa exato: o veto no caso claro.


In [5]:
cols = '''
    match_probability,
    mae_dif,
    nome_pessoa,
    data_par,
    ROUND(jw_mae, 3) AS jw_mae,
    nome_censo,
    nome_cpf,
    mae_censo,
    mae_cpf,
    dob_censo,
    dob_cpf
'''

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif IN ('mesmas_palavras_ordem', 'mesmas_palavras', 'contem_sem_prefixo')
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif = 'parecidas' AND nome_pessoa = 'exato'
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif = 'sem_palavra' AND nome_pessoa = 'exato'
ORDER BY match_probability DESC
LIMIT 20
''').df())


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.993086,contem_sem_prefixo,outro,exata,0.653,FERNANDA OLIVEIRA FEREIA,FERNANDA OLIVEIRA FEREIRA,MARIA OLIVEIRA,MARIA CRECENCIA CONCEICAO OLIVEIRA,1987-01-24,1987-01-24
1,0.991844,contem_sem_prefixo,outro,exata,0.736,DOMIGAS IZADORA SIUVA LIMA,DOMINGAS IZADORA SIUVA LIMA,RAQUEL DIAS SILVA,ANTONIA RAQUEL DIAS SILVA,2013-03-31,2013-03-31
2,0.991501,contem_sem_prefixo,pontas,exata,0.629,ABRAU LUKAS KATANHEDI PEREIRA,ABRAU LUKAS KANTANHEDI PEREIRA,GORETH PEREIRA,MARIA GORETH CANTANHEDE PEREIRA,1994-03-19,1994-03-19
3,0.991463,contem_sem_prefixo,outro,exata,0.609,ATENOR RODRIGUIS MARINHU,ANTENOR RODRIGUIS MARINHU,EVA MARINHO,EVA RODRIGUES MARINHO,1975-10-14,1975-10-14
4,0.991444,contem_sem_prefixo,outro,exata,0.736,UELIGUITON OLIVEIRA,UELINGUITON OLIVEIRA,MARIA OLIVEIRA,ANTONIA MARIA OLIVEIRA,1977-09-11,1977-09-11
5,0.991444,contem_sem_prefixo,outro,exata,0.666,RAIZA RAIMUNDA LOBATU PINHEIRU,RAISA RAIMUNDA LOBATU PINHEIRU,SONIA LOBATO,SONIA NAZARE SILVA LOBATO,1992-07-03,1992-07-03
6,0.991425,contem_sem_prefixo,outro,exata,0.667,RANSISKU MORAIS SIUVA,FRANSISKU MORAIS SIUVA,GRACINDA SILVA,GRACINDA MARIA TEIXEIRA MORAIS SILVA,1982-04-27,1982-04-27
7,0.991387,contem_sem_prefixo,pontas,exata,0.743,IURI RENAN FONTINELI SANTUS,IURI RENAN FONTENELI SANTUS,HOSANA SOUSA SANTOS,ANTONIA HOSANA SOUSA SANTOS,2013-10-12,2013-10-12
8,0.991329,contem_sem_prefixo,pontas,exata,0.542,MELISA RAFAELA SIUVA SIUVA,MELISA RAFAELI SIUVA SIUVA,IRLENE SILVA,MARIA IRLENE CONCEICAO SILVA,2016-07-03,2016-07-03
9,0.991253,contem_sem_prefixo,outro,exata,0.712,KOSMU RIBEIRU LEITI,KOSMI RIBEIRU LEITI,LUCIA RIBEIRO,MARIA LUCIA RIBEIRO LEITE,1978-09-27,1978-09-27


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.921978,parecidas,exato,diferente,0.925,JOZELIA SANTUS KONSEIKAU,JOZELIA SANTUS KONSEIKAU,MARIA CONCEICAO MELO,MARIA CONCEICAO SANTOS,1987-01-12,1988-10-12
1,0.920652,parecidas,exato,diferente,0.992,KAILANI SANTUS NASIMENTU,KAILANI SANTUS NASIMENTU,ELIANA SANTOS NASCIMNETO,ELIANA SANTOS NASCIMENTO,2004-03-27,2003-03-27
2,0.910142,parecidas,exato,diferente,0.938,RAIUSON SIUVA KARDOZU,RAIUSON SIUVA KARDOZU,MAGUINOLIA SILVA CARDOSO,MAGNOLIA SILVA CARDOSO,1984-02-02,1994-02-04
3,0.908198,parecidas,exato,diferente,0.921,FRANSISKA MIXELI VIEIRA MIRANDA,FRANSISKA MIXELI VIEIRA MIRANDA,IVONEIDE GOMES VIEIRA,IVANEIDE GOMES VIEIRA,2001-05-12,2001-05-13
4,0.908198,parecidas,exato,diferente,0.983,ELIZON SERA MARKIS,ELIZON SERA MARKIS,MARCIARA CARVALHO SERRA,MARCYARA CARVALHO SERRA,2007-09-14,2007-09-13
5,0.908198,parecidas,exato,diferente,0.974,ARIANI KRISTINA KARDOZU RIBEIRU,ARIANI KRISTINA KARDOZU RIBEIRU,ADEJANE CRISTINA ARAUJO CARDOSO,ADJANY CRISTINA ARAUJO CARDOSO,2007-11-29,2007-11-28
6,0.908198,parecidas,exato,diferente,0.952,LUSIENI MIRANDA MARTINS,LUSIENI MIRANDA MARTINS,MARIA OLINDA GOLCALVES MIRANDA,MARIA OLINDA GONCALVES MIRANDA,1983-02-03,1983-02-01
7,0.908198,parecidas,exato,diferente,0.975,JOZI AUBERTU RAPOZU SOUZA,JOZI AUBERTU RAPOZU SOUZA,LOURENCA RAPOSO SOUSA,LORENCIA RAPOSO SOUSA,1967-11-30,1967-11-29
8,0.908198,parecidas,exato,diferente,0.987,MARIA SIBELI OLIVEIRA SANTUS,MARIA SIBELI OLIVEIRA SANTUS,ANTONIA OLIVEIRA SOUZA CONCEICAO,ANTONIA OLIVEIRA SOUSA CONCEICAO,2018-02-19,2018-02-18
9,0.908198,parecidas,exato,diferente,0.962,ENDERSON RIKEUMI SIUVA SANTUS,ENDERSON RIKEUMI SIUVA SANTUS,ROSANGELA BIRA SILVA SANTOS,ROSANGELA LIRA SILVA SANTOS,2009-05-09,2009-05-08


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.984569,sem_palavra,exato,diferente,0.532,ISTEFANI BLENDA PEREIRA AMORIN,ISTEFANI BLENDA PEREIRA AMORIN,LUCIBELA ALVES JESUS,MARIA DAUVERCI PEREIRA,2008-08-08,2000-08-08
1,0.984569,sem_palavra,exato,diferente,0.497,TALITA EMILI OLIVEIRA REGU,TALITA EMILI OLIVEIRA REGU,EDNA SOUSA REGO,VERA ALICE OLIVEIRA DIAS,2013-11-13,2003-11-13
2,0.984569,sem_palavra,exato,diferente,0.696,NAIRA MONIKI PEREIRA SOUZA,NAIRA MONIKI PEREIRA SOUZA,MARIANA FERREIRA,FRANCINILDE SIQUEIRA PEREIRA,2015-04-14,2012-04-14
3,0.984569,sem_palavra,exato,diferente,0.542,DANILU OLIVEIRA MOURA,DANILU OLIVEIRA MOURA,NATALIA SANTOS OLIVEIRA,RAIMUNDA FERREIRA MOURA,2003-08-02,2005-08-02
4,0.984569,sem_palavra,exato,diferente,0.622,MARKUS ANTONIU MARINHU SA,MARKUS ANTONIU MARINHU SA,MARIA LUCIA PEREIRA,CRISTIANE LEITE MARINHO,1999-01-02,1997-01-02
5,0.984569,sem_palavra,exato,diferente,0.635,PAULU RIKARDU SEGUINS BARBOZA,PAULU RIKARDU SEGUINS BARBOZA,ELISABETH REGO BARBOSA,MARIA JESUS RABELO SEGUINS,2011-12-02,2001-12-02
6,0.984569,sem_palavra,exato,diferente,0.587,MARIA ELAINI SOUZA MORAIS,MARIA ELAINI SOUZA MORAIS,ANALIA DUARTE FRANCA,ADELAIDE RODRIGUES SOUSA MORAIS,2001-08-29,2008-08-29
7,0.984569,sem_palavra,exato,diferente,0.603,IANA GABRIELI RODRIGUIS OLIVEIRA,IANA GABRIELI RODRIGUIS OLIVEIRA,WELITA OLIVEIRA SOUSA,MARIA RODRIGUES GOMES,2013-10-20,2015-10-20
8,0.984569,sem_palavra,exato,diferente,0.509,BRUNA VITORIA AUVIS TEIXEIRA SOUZA,BRUNA VITORIA AUVIS TEIXEIRA SOUZA,MARINALVA BARBOSA SANTOS,RAQUEL ALVES TEIXEIRA,2002-09-09,2005-09-09
9,0.984569,sem_palavra,exato,diferente,0.545,MARIA GUIRASIELI SOUZA LIMA,MARIA GUIRASIELI SOUZA LIMA,MARTHA ARIELE RODRIGUEES SANTOS SOUSA,GLEYSON LIMA SILVA,2016-12-26,2014-12-26


In [6]:
con.close()
